In [5]:
from pathlib import Path
import duckdb

cwd = Path.cwd()
root = cwd.parent if cwd.name == 'notebooks' else cwd
db_path = root / "data" / "fdic.duckdb"
assert db_path.exists(), f"[duckdb_err]: path {db_path} not found"

conn = duckdb.connect(str(db_path), read_only=True)


In [6]:
df = conn.sql("SELECT * from fct_bank_quarter_training").df()
df.shape, df.head()

((693162, 20),
    cert report_date bank_quarter_id  failure_in_year  \
 0     9  2003-09-30      9_20030930                0   
 1    14  2003-09-30     14_20030930                0   
 2    35  2003-09-30     35_20030930                0   
 3    39  2003-09-30     39_20030930                0   
 4    41  2003-09-30     41_20030930                0   
 
    total_risk_based_capital_pct  nonperforming_assets_to_assets_pct  \
 0                     14.265452                            0.481639   
 1                     12.774630                            0.009425   
 2                     14.108060                            0.432329   
 3                     16.126012                            2.742972   
 4                     22.463488                            1.508979   
 
    loan_loss_allowance_to_loans_pct   roa_pct  net_interest_margin_pct  \
 0                          1.529200  1.154603                 3.960854   
 1                          0.891021  0.478641           

In [14]:
import pandas as pd

df['report_date'] = pd.to_datetime(df['report_date'])

train_split_dt = pd.Timestamp('2008-12-31')
test_split_dt = pd.Timestamp('2010-01-01')

train_df = df[df['report_date'] <= train_split_dt].copy()
test_df = df[df['report_date'] >= test_split_dt].copy()

print(f"Train dataframe: {train_df.shape}; Test dataframe: {test_df.shape}")

print(df['failure_in_year'].sum())
print(train_df['failure_in_year'].sum())
print(test_df['failure_in_year'].sum())


Train dataframe: (291719, 20); Test dataframe: (368527, 20)
2306
530
1109


In [12]:
feature_columns = [
    'total_risk_based_capital_pct', 
    'nonperforming_assets_to_assets_pct', 
    'loan_loss_allowance_to_loans_pct', 
    'roa_pct', 
    'net_interest_margin_pct', 
    'efficiency_ratio_pct', 
    'brokered_deposits_pct', 
    'core_deposits_pct', 
    'fhlb_advances_to_assets_pct', 
    'construction_concentration_pct', 
    'leverage_ratio_pct', 
    'core_deposits_pct_4q_chg', 
    'nonperforming_assets_to_assets_pct_4q_chg', 
    'roa_pct_4q_chg', 
    'total_risk_based_capital_pct_4q_chg', 
    'leverage_ratio_pct_4q_chg'
]

x_train = train_df[feature_columns]
y_train = train_df['failure_in_year']

x_test = test_df[feature_columns]
y_test = test_df['failure_in_year']


x_train_null_cnt = x_train.isnull().sum()
print(x_train_null_cnt)


total_risk_based_capital_pct                   638
nonperforming_assets_to_assets_pct               0
loan_loss_allowance_to_loans_pct                 0
roa_pct                                          0
net_interest_margin_pct                          0
efficiency_ratio_pct                             0
brokered_deposits_pct                         3492
core_deposits_pct                                0
fhlb_advances_to_assets_pct                    622
construction_concentration_pct                5787
leverage_ratio_pct                               0
core_deposits_pct_4q_chg                     43589
nonperforming_assets_to_assets_pct_4q_chg    43589
roa_pct_4q_chg                               43589
total_risk_based_capital_pct_4q_chg          44139
leverage_ratio_pct_4q_chg                    43589
dtype: int64
